In [ ]:
import pandas as pd
import json
from datetime import datetime
pd.set_option('display.max_columns', None)


In [ ]:
df_institution_raw = catalog.load('raw/openalex/institution#parquet')

# Profiling

In [ ]:
df_institution_raw.columns

In [ ]:
df_institution_raw.head(3)

# Nodo

In [ ]:
def openalex_load_institution(df_institution_raw):

    expected_columns = [
        'id',
        'ror',
        'display_name',
        'country_code',
        'type',
        'type_id',
        # 'lineage',
        'homepage_url',
        'image_url',
        'image_thumbnail_url',
        'display_name_acronyms',
        # 'display_name_alternatives',
        # 'repositories',
        'works_count',
        'cited_by_count',
        'summary_stats',
        'ids',
        'geo',
        'international',
        # 'associated_institutions',
        # 'counts_by_year',
        'roles',
        'topics',
        'topic_share',
        'x_concepts',
        'is_super_system',
        'works_api_url',
        'updated_date',
        'created_date',
        'extract_datetime'
    ]
    df_institution = df_institution_raw.loc[:,expected_columns].reset_index(drop=True).copy()

    # Agregar columnas faltantes con NaN
    for col in expected_columns:
        if col not in df_institution.columns:
            df_institution[col] = pd.NA

    # ids
    df_ids = pd.json_normalize(df_institution['ids']).reset_index(drop=True)
    df_institution = pd.concat([df_institution, df_ids], axis=1)
    df_institution.drop(columns=['ids'], inplace=True)    

    # summary_stats
    df_summary_stats = pd.json_normalize(df_institution['summary_stats']).reset_index(drop=True)
    df_summary_stats.rename(columns=lambda col: f'summary_stats.{col}', inplace=True)
    df_institution = pd.concat([df_institution, df_summary_stats], axis=1)
    df_institution.drop(columns=['summary_stats'], inplace=True)    

    # geo
    df_geo = pd.json_normalize(df_institution['geo']).reset_index(drop=True)
    df_geo.rename(columns=lambda col: f'geo.{col}', inplace=True)
    df_institution = pd.concat([df_institution, df_geo], axis=1)
    df_institution.drop(columns=['geo'], inplace=True)    

    # international
    df_international = pd.json_normalize(df_institution['international']).reset_index(drop=True)
    df_international.rename(columns=lambda col: f'international.{col}', inplace=True)
    df_institution = pd.concat([df_institution, df_international], axis=1)
    df_institution.drop(columns=['international'], inplace=True)    

    df_institution['_load_datetime'] = pd.to_datetime(datetime.today())

    # Convertir tipos de datos automáticamente
    df_institution = df_institution.convert_dtypes()

    return df_institution


## Ejecuto Nodo

In [ ]:
df_institution = openalex_load_institution(df_institution_raw)

# Resultados

In [ ]:
df_institution